# Video Tokenizer Training (Spacetime Vector Quantized Variational Autoencoder)

In [1]:
import torch 
import lpips

from torch.utils.data import DataLoader
from torchvision.datasets import UCF101

import lightning as L
from spacetime.models.tokenizers import STVQVae

import wandb

In [2]:
wandb.login()

wandb: Currently logged in as: aryaman-pandya (aryaman-pandya-99) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Lightning module

We will use pytorch lightning to reduce boiler plate (there's a lot in previous notebooks, despite the centralized modules in `/src`)

In [ ]:
class STVQVaeModule(L.LightningModule):
    def __init__(
        self,
        num_heads,
        d_model,
        num_layers,
        d_linear,
        codebook_size,
        codebook_dim,
        patch_size,
        frame_height,
        frame_width,
        num_frames,
        num_linear_layers=2,
        num_groups=8,
        dropout=0.1,
        beta=0.25
    ):
        super().__init__()
        self.model = STVQVae(
            num_heads=num_heads,
            d_model=d_model,
            num_layers=num_layers,
            d_linear=d_linear,
            codebook_size=codebook_size,
            codebook_dim=codebook_dim,
            patch_size=patch_size,
            frame_height=frame_height,
            frame_width=frame_width,
            num_frames=num_frames,
            num_linear_layers=num_linear_layers,
            num_groups=num_groups,
            dropout=dropout
        )
        self.beta = beta
        self.example_clip = None
        self.example_recon = None

        self.lpips_metric = lpips.LPIPS(net="vgg")
        self.lpips_metric.eval()
        for p in self.lpips_metric.parameters():
            p.requires_grad = False

    def forward(self, inputs):
        return self.model(inputs)
    
    def configure_optimizers(self):
        return torch.optim.AdamW(self.model.parameters(), lr=3e-4)

    def training_step(self, batch, batch_idx):
        x, _ = batch
        x_pred, z_e, z_quantized = self(x)
        recon_loss = torch.nn.functional.mse_loss(x_pred, x)
        codebook_loss = torch.nn.functional.mse_loss(z_quantized, z_e.detach())
        commit_loss = torch.nn.functional.mse_loss(z_e, z_quantized.detach())
        loss = recon_loss + codebook_loss + (self.beta * commit_loss)

        self._log_losses(loss, recon_loss, codebook_loss, commit_loss, is_training=True)

        if batch_idx == 0:
            self.example_clip = x[:1].detach().cpu()
            self.example_recon = x_pred[:1].detach().cpu()
        return loss

    def validation_step(self, batch, batch_idx):
        x, _ = batch
        x_pred, z_e, z_quantized = self(x)
        recon_loss = torch.nn.functional.mse_loss(x_pred, x)
        codebook_loss = torch.nn.functional.mse_loss(z_quantized, z_e.detach())
        commit_loss = torch.nn.functional.mse_loss(z_e, z_quantized.detach())
        loss = recon_loss + codebook_loss + (self.beta * commit_loss)

        self._log_losses(loss, recon_loss, codebook_loss, commit_loss, is_training=False)

        with torch.no_grad():
        # LPIPS expects inputs in [-1,1]; convert if you’re in [0,1]
            B, C, F, H, W = x.shape
            to_lpips = lambda t: ((t * 2.0) - 1.0).reshape(B * F, C, H, W)
            lpips_val = self.lpips_metric(to_lpips(x_pred), to_lpips(x)).mean()
        self.log("val_lpips", lpips_val, prog_bar=False, logger=True)
        if wandb.run is not None:
            wandb.log({"val_lpips": lpips_val.item()}, step=self.global_step)
        return loss
    
    def on_validation_epoch_end(self):
        if self.example_clip is None or wandb.run is None:
            return
        clip = (self.example_clip.clamp(0, 1) * 255).to(torch.uint8)
        recon = (self.example_recon.clamp(0, 1) * 255).to(torch.uint8)
        video = torch.cat([clip, recon], dim=4)       # or dim=2/3, whichever you chose
        video = video.squeeze(0).permute(1, 0, 2, 3)  # (F, C, H, W)
        wandb.log(
            {
                "recon_video": wandb.Video(
                    video.squeeze(0), fps=4, format="mp4"
                )
            },
            step=self.global_step,
        )
        self.example_clip = None
        self.example_recon = None
    
    def _log_losses(self, loss, recon_loss, codebook_loss, commit_loss, is_training=True):
        prefix = "train" if is_training else "val"
        log_on_step = True if is_training else False
        log_on_epoch = True

        # Lightning logging
        self.log(f"{prefix}_loss", loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=True, logger=True)
        self.log(f"{prefix}_recon_loss", recon_loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=False, logger=True)
        self.log(f"{prefix}_codebook_loss", codebook_loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=False, logger=True)
        self.log(f"{prefix}_commit_loss", commit_loss, on_step=log_on_step, on_epoch=log_on_epoch, prog_bar=False, logger=True)

        # Weights & Biases logging
        if wandb.run is not None:
            wandb.log({
                f"{prefix}_loss": loss.item(),
                f"{prefix}_recon_loss": recon_loss.item(),
                f"{prefix}_codebook_loss": codebook_loss.item(),
                f"{prefix}_commit_loss": commit_loss.item(),
            }, step=self.global_step)

    

## Load UCF101 Action Recognition dataset 

We use the UCF101 dataset which contains 13,320 videos from 101 action categories. This dataset is commonly used for benchmarking video action recognition models, such as basketball shooting, biking, diving, golf swinging, horse riding, and playing musical instruments.


We created a subset of UCF101 with only 10 classes for faster experimentation. The selected classes are:
ApplyEyeMakeup, ApplyLipstick, Archery, BabyCrawling, BalanceBeam, BandMarching, BaseballPitch, Basketball, BasketballDunk, and BenchPress.

In [4]:
train_dataset = UCF101(
    root='./data/UCF-101-downsized',
    annotation_path='./data/ucfTrainTestlist',
    frames_per_clip=8,
    step_between_clips=8,  # non overlapping clips
    train=True,
)

test_dataset = UCF101(
    root='./data/UCF-101-downsized',
    annotation_path='./data/ucfTrainTestlist',
    frames_per_clip=8,
    step_between_clips=8,
    train=False,
)

print(f"Downsized train dataset size: {len(train_dataset)} clips")
print(f"Downsized test dataset size: {len(test_dataset)} clips")

  0%|          | 0/86 [00:00<?, ?it/s]

/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
  0%|          | 0/86 [00:00<?, ?it/s]/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
100%|██████████| 86/86 [00:29<00:00,  2.92it/s]

Downsized train dataset size: 17561 clips
Downsized test dataset size: 6464 clips


In [5]:
import torch.nn.functional as F


def collate_ucf101(batch):
    # batch: list of (video, label, index) where label is detection labels 
    # and index is the index of the class for recognition
    xs, ys = [], []
    for v, _, l in batch:
        # v: T, H, W, C  (uint8)
        v = v.permute(0, 3, 1, 2)            # -> T, C, H, W
        v = v.float() / 255.0
        v = F.interpolate(v, size=(224, 224), mode='bilinear', align_corners=False)  # resize frames
        v = v.permute(1, 0, 2, 3).contiguous()  # -> C, F, H, W
        xs.append(v.clone())                  # new storage
        ys.append(int(l))
    return torch.stack(xs, 0), torch.tensor(ys, dtype=torch.long)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=8,
    collate_fn=collate_ucf101,
    pin_memory=True,
)

test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_ucf101)

In [ ]:
params = {
    "num_heads": 6,
    "d_model": 384,
    "num_layers": 3,
    "d_linear": 1536,
    "codebook_size": 1024,
    "codebook_dim": 128,
    "patch_size": 8,
    "frame_height": 224,
    "frame_width": 224,
    "num_frames": 8,
    "num_linear_layers": 2,
    "num_groups": 8,
    "dropout": 0.1,
    "max_epochs": 10,
    "precision": 16,
    "batch_size": 4,
}

wandb.init(
    project="spacetime",
    name=f"stvqvae_patch{params['patch_size']}_layers{params['num_layers']}",
    config=params,
)

In [ ]:

lightning_timesformer = STVQVaeModule(
    num_heads=params["num_heads"],
    d_model=params["d_model"],
    num_layers=params["num_layers"],
    d_linear=params["d_linear"],
    codebook_size=params["codebook_size"],
    codebook_dim=params["codebook_dim"],
    patch_size=params["patch_size"],
    frame_height=params["frame_height"],
    frame_width=params["frame_width"],
    num_frames=params["num_frames"],
    num_linear_layers=params["num_linear_layers"],
    num_groups=params["num_groups"],
    dropout=params["dropout"],
)

wandb.watch(lightning_timesformer, log="gradients", log_freq=100)

trainer = L.Trainer(max_epochs=5, precision=16)

"""
# sanity check: 

trainer = L.Trainer(
    max_epochs=1,
    limit_train_batches=1,
    limit_val_batches=1,
    fast_dev_run=False,
    precision=16
)
"""
trainer.fit(model=lightning_timesformer, train_dataloaders=train_dataloader, val_dataloaders=test_dataloader)

wandb.finish()

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/lightning/fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try install

Loading model from: /home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/aryamanpandya/spacetime/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA A100 80GB PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RAN

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x761360dae980>> (for post_run_cell), with arguments args (<ExecutionResult object at 761360daf250, execution_count=7 error_before_exec=None error_in_exec=1 info=<ExecutionInfo object at 761360daed10, raw_cell="
lightning_timesformer = STVQVaeModule(
    num_he.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bcoder-vscode.coder.aks-prod-coder-swedencentral.azr.wayve.ai--aryamanpandya--ap-ml-exp.workspace/home/aryamanpandya/spacetime/src/spacetime/nbs/tokenizer.ipynb#X12sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.s

BrokenPipeError: [Errno 32] Broken pipe

socket.send() raised exception.
socket.send() raised exception.


socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.send() raised exception.
socket.s

## Notes 

- Patch sizes of 8 and 16 lead to coarse reconstructions. (improving as we drop the patch size)